# Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/notebooks/intro/08_optimization.ipynb)

Official API intro to finite-dimensional NLPs: `MathematicalProgram` and
`Optimizer` (SciPy / Ipopt backends).

**Scripts for depth:** `examples/scripts/optimization/`

**See also:** [`intro/09_planning.ipynb`](09_planning.ipynb) for trajectory optimization on systems.


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")


In [ ]:
import numpy as np

from minilink.optimization.mathematical_program import MathematicalProgram
from minilink.optimization.optimizer import Optimizer

## Unconstrained quadratic

$$
\min_{z \in \mathbb{R}^{3}} \; \frac{1}{2} \, \lVert z - \bar{z} \rVert_2^2
$$

The minimum is $z^\star = \bar{z}$ (closed form). The analytic gradient is $\nabla J(z) = z - \bar{z}$.

In [ ]:
z_bar = np.array([1.0, -0.5, 2.0])
z0 = np.zeros_like(z_bar)


def J(z: np.ndarray):
    r = z - z_bar
    return 0.5 * (r @ r)


def grad(z: np.ndarray) -> np.ndarray:
    return z - z_bar


unc = MathematicalProgram(n_z=3, J=J, grad_J=grad)
out = Optimizer(
    unc,
    z0=z0,
    method="scipy_slsqp",
    # compile_backend="jax",
    options={"ftol": 1e-12},
).solve(disp=True)

## Convex QP (quadratic objective, linear inequalities)

Standard **QP** form (here with symmetric positive semidefinite $Q$):

$$
\min_{z} \; \frac{1}{2}\, \lVert z \rVert_2^2
\quad \text{s.t.} \quad z_1 \geq 0,\; z_2 \geq 0,\; z_1 + z_2 - 1 \geq 0 .
$$

The unconstrained minimum is $0$ at $z=0$, which violates $z_1+z_2 \geq 1$. The constrained minimum is the point on the line $z_1+z_2=1$ in the first quadrant **closest to the origin**: $z^\star = [\tfrac{1}{2},\,\tfrac{1}{2}]^\top$.

In [ ]:
Q2 = np.eye(2)


def J_qp(z: np.ndarray):
    return 0.5 * (z @ Q2 @ z)


def grad_qp(z: np.ndarray) -> np.ndarray:
    return Q2 @ z


def g_qp(z: np.ndarray) -> np.ndarray:
    return np.array([z[0], z[1], z[0] + z[1] - 1.0])


def jac_qp(z: np.ndarray) -> np.ndarray:
    return np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])


qp = MathematicalProgram(
    n_z=2,
    J=J_qp,
    grad_J=grad_qp,
    g=g_qp,
    jac_g=jac_qp,
)
out = Optimizer(
    qp,
    z0=np.array([0.6, 0.6]),
    method="scipy_slsqp",
    # compile_backend="jax",
    options={"ftol": 1e-12},
).solve(disp=True)

## Optimizer backends

`Optimizer(method=...)` selects SciPy SLSQP by default; Ipopt is available when
`cyipopt` is installed.


In [ ]:
from minilink.optimization import Optimizer
import inspect
print("Optimizer:", Optimizer)
sig = None
try:
    sig = inspect.signature(Optimizer.__init__)
except Exception:
    pass
print(sig)
print("Demo with plots: examples/scripts/optimization/demo_optim_plot.py")
